In [2]:
import requests
from bs4 import BeautifulSoup

import regex as re
from datetime import datetime
import time
import random

from urllib.parse import urljoin

In [3]:
# Extraer todas las URLs de noticias, es diferente a la función del resto de periódicos porque en la web se encuentran
# las url relativas y no absolutas, lo cual complica el trabajo
def extraer_urls(periodico, seccion, extension):
    base = f"https://{periodico}"
    urls = []

    url = f"{base}/{extension}"
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    patron = re.compile(rf"/{seccion}/[^/]+\.html$")

    for enlace in soup.find_all("a"):
        href = enlace.get("href")
        if not href:
            continue

        if patron.match(href):
            absoluta = urljoin(base, href)
            if absoluta not in urls:
                urls.append(absoluta)

    return urls

In [4]:
def extraer_noticia(url, nombre_seccion, periodico):
    
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Título
    titulo = soup.find("h1")
    titulo = titulo.get_text(strip=True) if titulo else None

    # Subtítulo
    subtitulo = soup.find("h2")
    subtitulo = subtitulo.get_text(" ", strip=True) if subtitulo else None

    # Texto del artículo
    cuerpo = soup.find("div", class_="c-detail__body")

    if cuerpo:
        parrafos = cuerpo.find_all("p")
        texto = "\n".join(p.get_text(" ", strip=True).replace("  ", " ") for p in parrafos)
    else:
        texto = ""
    
    fecha_actual = datetime.now().strftime("%d-%m-%Y")

    noticia = {
            "Link": url,
            "Periódico": periodico,
            "Fecha": fecha_actual,
            "Título": titulo,
            "Subtítulo": subtitulo if subtitulo else None,
            "Categoría": nombre_seccion,
            "Contenido": texto
        }
    return noticia


In [5]:
url = "https://www.huffingtonpost.es/life/cultura/elvira-minguez-f202605.html"
noticia = extraer_noticia(url, "cultura", "Huffpost")
print(" ========== ========== ========= ======== =========")
print("------------ ###### TITULO ###### ---------")
print(noticia["Título"])
print("/n")
print("------------ ###### SUBTITULO ###### ---------")
print(noticia["Subtítulo"])
print("/n")
print("------------ ###### CONTENIDO ###### ---------")
print(noticia["Contenido"])
print("/n")

 ========== ========== ========= ======== =========
------------ ###### TITULO ###### ---------
Elvira Mínguez: "Como las cosas se tuerzan, las primeras que vamos a perder somos nosotras"
/n
------------ ###### SUBTITULO ###### ---------
Hablamos con la actriz, directora y escritora sobre su irrupción por todo lo alto en el mundo de la literatura con su segunda novela, 'La educación del monstruo', Premio Primavera de Novela.
/n
------------ ###### CONTENIDO ###### ---------
Desde que el pasado 27 de febrero se hiciese público el fallo del Premio Primavera de Novela, la actriz Elvira Mínguez ha vivido auténticas semanas de locura. Primero, porque solo unos días después de que se anunciase que su novela La educación del monstruo (Ed. Espasa) se había alzado con este galardón literario, aparecía radiante en la alfombra roja de los Goya, pues su interpretación de una cocinera republicana en La cena había sido reconocida con la nominación al Goya a la Mejor interpretación femenina de repart

In [6]:
def scrappeo_seccion(periodico, seccion, nombre_seccion, extension):
    urls = extraer_urls(periodico=periodico, seccion=seccion, extension=extension)
    noticias = []
    i=1
    for url in urls:
        noticias.append(extraer_noticia(url, nombre_seccion=nombre_seccion, periodico=periodico))
        time.sleep(1)
        print(f"Noticia {i} de {len(urls)}")
        i+=1
    return noticias

url_base = "https:/"
periodico = "www.eldiario.es"
seccion = "internacional" 

# noticias = scrappeo_seccion(periodico="www.eldiario.es", seccion="internacional")

In [7]:
'''for noticia in noticias:
    print(" ========== ========== ========= ======== =========")
    print("------------ ###### TITULO ###### ---------")
    print(noticia["Título"])
    print("/n")
    print("------------ ###### SUBTITULO ###### ---------")
    print(noticia["Subtítulo"])
    print("/n")
    print("------------ ###### CONTENIDO ###### ---------")
    print(noticia["Contenido"])
    print("/n")'''

'for noticia in noticias:\n    print(" ========== ========== ========= ======== =========")\n    print("------------ ###### TITULO ###### ---------")\n    print(noticia["Título"])\n    print("/n")\n    print("------------ ###### SUBTITULO ###### ---------")\n    print(noticia["Subtítulo"])\n    print("/n")\n    print("------------ ###### CONTENIDO ###### ---------")\n    print(noticia["Contenido"])\n    print("/n")'

In [8]:
# GUARDADO DE LAS NOTICIAS EN UN JSON
import json
import os

def guardar_noticias(noticias, archivo_json):
    
    # Si el archivo existe, cargar su contenido
    if os.path.exists(archivo_json):
        with open(archivo_json, "r", encoding="utf-8") as f:
            datos_existentes = json.load(f)
    else:
        datos_existentes = []

    # Añadir las nuevas noticias
    datos_existentes.extend(noticias)

    # Guardar todo de nuevo
    with open(archivo_json, "w", encoding="utf-8") as f:
        json.dump(datos_existentes, f, ensure_ascii=False, indent=4)

# guardar_noticias(noticias=noticias)


In [ ]:
# Por si acaso cambian las rutas locales, lo ponemos como una variable y así solo hay que cambiarlo una vez
ruta_guardado = "./../../data/elhuffpost.json"

# Por cómo es la página del periódico, tenemos que escrapear a parte la página inicial de la sección de después el
# resto de páginas, pero se hace con un bucle
noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Virales",
                            seccion="virales", 
                            extension="virales#int=submenu_3")

guardar_noticias(noticias=noticias, 
                 archivo_json=ruta_guardado)

for i in range(2, 6):
    noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Virales",
                            seccion="virales", 
                            extension=f"virales/{i}")

    guardar_noticias(noticias=noticias, 
                    archivo_json=ruta_guardado)

time.sleep(30)

Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia 1 de 19
Noticia 2 de 19
Noticia 3 de 19
Noticia 4 de 19
Noticia 5 de 19
Noticia 6 de 19
Noticia 7 de 19
Noticia 8 de 19
Noticia 9 de 19
Noticia 10 de 19
Noticia 11 de 19
Noticia 12 de 19
Noticia 13 de 19
Noticia 14 de 19
Noticia 15 de 19
Noticia 16 de 19
Noticia 17 de 19
Noticia 18 de 19
Noticia 19 de 19
Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia 1 de 20
Noticia 

In [19]:
# Ahora scrapeamos la sección de CULTURA
# Esta sección se puede scrapear entera con un bucle

for i in range(1, 6):
    noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Cultura",
                            seccion="life/cultura", 
                            extension=f"life/cultura/{i}")

    guardar_noticias(noticias=noticias, 
                     archivo_json=ruta_guardado)

time.sleep(30)

Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia

In [20]:
# Vamos con la sección de política, pues el periódico no tiene sección nacional, esta es la más parecida

# Por cómo es la página del periódico, tenemos que escrapear a parte la página inicial de la sección de después el
# resto de páginas, pero se hace con un bucle
noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Nacional",
                            seccion="politica", 
                            extension="politica#int=ham_1")

guardar_noticias(noticias=noticias, 
                 archivo_json=ruta_guardado)

for i in range(2, 6):
    noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Nacional",
                            seccion="politica", 
                            extension=f"politica/{i}")

    guardar_noticias(noticias=noticias, 
                    archivo_json=ruta_guardado)

time.sleep(30)

Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20
Noticia 1 de 19
Noticia 2 de 19
Noticia 3 de 19
Noticia 4 de 19
Noticia 5 de 19
Noticia 6 de 19
Noticia 7 de 19
Noticia 8 de 19
Noticia 9 de 19
Noticia 10 de 19
Noticia 11 de 19
Noticia 12 de 19
Noticia 13 de 19
Noticia 14 de 19
Noticia 15 de 19
Noticia 16 de 19
Noticia 17 de 19
Noticia 18 de 19
Noticia 19 de 19
Noticia 1 de 19
Noticia 2 de 19
Noticia 3 de 19
Noticia 4 de 19
Noticia 5 de 19
Noticia 6 de 19
Noticia 7 de 19
Noticia 8 de 19
Noticia 9 de 19
Noticia 10 de 19
Noticia 11 de 19
Noticia 12 de 19
Noticia 13 de 19
Noticia 14 de 19
Noticia 15 de 19
Noticia 16 de 19
Noticia 17 de 19
Noticia 18 de 19
Noticia 19 de 19
Noticia 1 de 19
Noticia 2 de 19
Noticia 3

In [ ]:
# Vamos con la sección global, pues el periódico no tiene sección internacional, esta es la más parecida

# Por cómo es la página del periódico, tenemos que escrapear a parte la página inicial de la sección de después el
# resto de páginas, pero se hace con un bucle
noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Internacional",
                            seccion="global", 
                            extension="global#int=submenu_2")

guardar_noticias(noticias=noticias, 
                 archivo_json=ruta_guardado)

for i in range(2, 6):
    noticias = scrappeo_seccion(periodico="www.huffingtonpost.es", 
                            nombre_seccion="Internacional",
                            seccion="global", 
                            extension=f"global/{i}")

    guardar_noticias(noticias=noticias, 
                    archivo_json=ruta_guardado)

time.sleep(30)

Noticia 1 de 16
Noticia 2 de 16
Noticia 3 de 16
Noticia 4 de 16
Noticia 5 de 16
Noticia 6 de 16
Noticia 7 de 16
Noticia 8 de 16
Noticia 9 de 16
Noticia 10 de 16
Noticia 11 de 16
Noticia 12 de 16
Noticia 13 de 16
Noticia 14 de 16
Noticia 15 de 16
Noticia 16 de 16
Noticia 1 de 19
Noticia 2 de 19
Noticia 3 de 19
Noticia 4 de 19
Noticia 5 de 19
Noticia 6 de 19
Noticia 7 de 19
Noticia 8 de 19
Noticia 9 de 19
Noticia 10 de 19
Noticia 11 de 19
Noticia 12 de 19
Noticia 13 de 19
Noticia 14 de 19
Noticia 15 de 19
Noticia 16 de 19
Noticia 17 de 19
Noticia 18 de 19
Noticia 19 de 19
Noticia 1 de 18
Noticia 2 de 18
Noticia 3 de 18
Noticia 4 de 18
Noticia 5 de 18
Noticia 6 de 18
Noticia 7 de 18
Noticia 8 de 18
Noticia 9 de 18
Noticia 10 de 18
Noticia 11 de 18
Noticia 12 de 18
Noticia 13 de 18
Noticia 14 de 18
Noticia 15 de 18
Noticia 16 de 18
Noticia 17 de 18
Noticia 18 de 18
Noticia 1 de 19
Noticia 2 de 19
Noticia 3 de 19
Noticia 4 de 19
Noticia 5 de 19
Noticia 6 de 19
Noticia 7 de 19
Noticia 8 de 1

In [ ]:
import pandas as pd
# Debido ciertos fallos en la programación de las funciones anteriores, los datos guardados en el JSON, no son correctos
# este es un chunck extra donde solventamos esos pequeños problemas para no ejecutar de nuevo todo el código
# 1. Importar el JSON a un DataFrame
ruta_guardado = "./../../data/elhuffpost.json"
df = pd.read_json(ruta_guardado, orient="records")

# 2. Reemplazar una palabra en la columna 'categoría' mediante un condicional
# Ejemplo: si categoría contiene "política", cambiarla por "Politica Nacional"
df.loc[df["Link"].str.contains("virales", case=False, na=False), "Categoría"] = "Cultura"

df['Categoría'] = df['Categoría'].apply(
    lambda x: "Internacional" if "Interacional" == x else x)

# 3. Eliminar registros donde el campo 'contenido' está vacío
df = df[df['Contenido'].str.strip() != ""]

# (Opcional) Reiniciar índice tras limpiar
df = df.reset_index(drop=True)


df.to_json(ruta_guardado, orient="records", force_ascii=False, indent=4)